In [1]:
import matplotlib
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.weight'] = 'normal'
from splitflow.figure_style import apply_house_style  # Helvetica 6 pt, Helvetica mathtext
apply_house_style(size=matplotlib.rcParams["font.size"])

In [2]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import random
import math

import pandas as pd

import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots
import fly_plot_lib.plot as fpl

import scipy.stats

/Users/florisvb/src/SplitFlowOptoTracking/.venv/lib/python3.11/site-packages/figurefirst/mpl_fig_to_figurefirst_svg.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
import sys
from pathlib import Path


In [4]:
from splitflow import unifying_algo_analysis_helper as uaah
from splitflow.set_zorder_functions import *

Using device: cpu


In [5]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

In [6]:
MARKERSIZE = 0.75
MODEL_LINE_COLOR = 'lightcoral'
TRANSLATION = True

# Colors match Figure 5: variable wind flash data light purple with 20 cm/s (the '28' file) highlighted; gray for sham data.
VARWIND_COLOR = '#c4aee0ff'
LOW_WIND_HIGHLIGHT_COLOR = '#a245ffff'  # 20 cm/s
LOW_WIND_HIGHLIGHT_SPEED = '28'  # file label for 20 cm/s
SHAM_COLOR = 'gray'

def get_color(condition, speed):
    if condition == 'sham':
        return SHAM_COLOR
    return LOW_WIND_HIGHLIGHT_COLOR if speed == LOW_WIND_HIGHLIGHT_SPEED else VARWIND_COLOR
CONDITIONS = ['flash', 'sham']  # flash on the left

In [7]:
if TRANSLATION:
    FIGURE_NAME = 'supplemental_unifying_analysis_variable_wind_unifying.svg'

In [8]:
def get_median_slope_and_axis_ratio(fname):
    df = pd.read_parquet(fname)

    slopes = []
    axis_ratios = []
    rotation = []

    for obj_id in df.obj_id_unique_event.unique():
        data = df[df.obj_id_unique_event==obj_id].copy()
        #slopes.append(data.slope.abs().median())

        data['slope_abs'] = data.slope.abs()
        slopes.append( uaa.get_weighted_value(data, col='slope_abs', use='rmse_affine') )

        axis_ratios.append( uaa.get_weighted_value(data, col='axis_ratio', use='rmse_affine') )

        data.rotation = data.rotation.abs()
        rotation.append( uaa.get_weighted_value(data, col='rotation', use='rmse_affine') )

            
    results = {'slopes': np.array(slopes)/0.01,
               'axis_ratios': np.array(axis_ratios),
               'rotation': np.array(rotation),
               #'percent_affine_improvements': np.array(percent_affine_improvements),
              }
    
    return results

In [9]:
df = pd.read_parquet(str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet')

In [10]:
df.fan_speed_percent.unique()

array([28., 23., nan, 20., 12., 40., 90.])

In [11]:
# Both conditions are plotted (see CONDITIONS)

In [12]:
class LabelToMetadata:
    def __init__(self):
        condition = 'flash'
        self.flash = {str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_12.0_translation' + str(TRANSLATION) + '.parquet': [1, 'gray', '12', 0], 
                      str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_20.0_translation' + str(TRANSLATION) + '.parquet': [2, 'gray', '20', 0.03],
                      str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_23.0_translation' + str(TRANSLATION) + '.parquet': [3, 'gray', '23', 0.1],
                      str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_28.0_translation' + str(TRANSLATION) + '.parquet': [4, '#a245ffff', '28', 0.2],
                      str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_40.0_translation' + str(TRANSLATION) + '.parquet': [5, 'gray', '40', 0.3],
                      str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_90.0_translation' + str(TRANSLATION) + '.parquet': [6, 'gray', '90', 0.6],
                       }
        condition = 'sham'
        self.sham = {str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_12.0_translation' + str(TRANSLATION) + '.parquet': [1, 'gray', '12', 0], 
                     str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_20.0_translation' + str(TRANSLATION) + '.parquet': [2, 'gray', '20', 0.03],
                     str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_23.0_translation' + str(TRANSLATION) + '.parquet': [3, 'gray', '23', 0.1],
                     str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_28.0_translation' + str(TRANSLATION) + '.parquet': [4, '#bd7bffff', '28', 0.2],
                     str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_40.0_translation' + str(TRANSLATION) + '.parquet': [5, 'gray', '40', 0.3],
                     str(Path('../../../Data/unifying_algo_results')) + '/varwind_N700_' + condition + '_90.0_translation' + str(TRANSLATION) + '.parquet': [6, 'gray', '90', 0.6],
                               }

        # Temporary
        self.sham = {str(Path('../../../Data/unifying_algo_results')) + '/var_wind_kevin_analyzed/' + 'var-wind-12-fan_0-cms_all-traj_align_SHAM' + '.parquet': [1, 'gray', '12', 0], 
                                 str(Path('../../../Data/unifying_algo_results')) + '/var_wind_kevin_analyzed/' + 'var-wind-20-fan_0-03-cms_all-traj_align_SHAM' + '.parquet': [2, 'gray', '20', 0.03],
                                 str(Path('../../../Data/unifying_algo_results')) + '/var_wind_kevin_analyzed/' + 'var-wind-23-fan_0-1-cms_all-traj_align_SHAM' + '.parquet': [3, 'gray', '23', 0.1],
                                 str(Path('../../../Data/unifying_algo_results')) + '/var_wind_kevin_analyzed/' + 'var-wind-28-fan_0-2-cms_all-traj_align_SHAM' + '.parquet': [4, '#bd7bffff', '28', 0.2],
                                 str(Path('../../../Data/unifying_algo_results')) + '/var_wind_kevin_analyzed/' + 'var-wind-40-fan_0-3-cms_all-traj_align_SHAM' + '.parquet': [5, 'gray', '40', 0.3],
                                 str(Path('../../../Data/unifying_algo_results')) + '/var_wind_kevin_analyzed/' + 'var-wind-90-fan_0-6-cms_all-traj_align_SHAM' + '.parquet': [6, 'gray', '90', 0.6],
                               }

In [13]:
label_to_metadata = LabelToMetadata()

In [14]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [15]:
label_to_metadata.flash, label_to_metadata.sham

({'../../../Data/unifying_algo_results/varwind_N700_flash_12.0_translationTrue.parquet': [1,
   'gray',
   '12',
   0],
  '../../../Data/unifying_algo_results/varwind_N700_flash_20.0_translationTrue.parquet': [2,
   'gray',
   '20',
   0.03],
  '../../../Data/unifying_algo_results/varwind_N700_flash_23.0_translationTrue.parquet': [3,
   'gray',
   '23',
   0.1],
  '../../../Data/unifying_algo_results/varwind_N700_flash_28.0_translationTrue.parquet': [4,
   '#a245ffff',
   '28',
   0.2],
  '../../../Data/unifying_algo_results/varwind_N700_flash_40.0_translationTrue.parquet': [5,
   'gray',
   '40',
   0.3],
  '../../../Data/unifying_algo_results/varwind_N700_flash_90.0_translationTrue.parquet': [6,
   'gray',
   '90',
   0.6]},
 {'../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-12-fan_0-cms_all-traj_align_SHAM.parquet': [1,
   'gray',
   '12',
   0],
  '../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-20-fan_0-03-cms_all-traj_align_SHAM.parquet':

In [16]:
feature = 'slopes'

for condition in CONDITIONS:
    metadata = label_to_metadata.__getattribute__(condition)
    show_labels = (condition == CONDITIONS[0])  # labels only on the left-most (flash) panel
    ax = layout.axes[('variable_wind_' + condition, 'slopes')]

    x_data_for_regression = []
    y_data_for_regression = []

    ticks = []
    ticklabels = []
    for key, val in metadata.items():
        fname = key
        results = get_median_slope_and_axis_ratio(fname)
        xix, _, label, speed = val
        color = get_color(condition, label)

        ticks.append(xix)
        ticklabels.append(label)

        hist_counts, hist_bins = np.histogram(results[feature], bins=20)
        fpl.scatter_box_violin_distribution(ax, xix, copy.copy(results[feature]), hist_counts, hist_bins, color=color, markersize=MARKERSIZE)

        if xix<=6:
            x_data_for_regression.append(xix)
            y_data_for_regression.append(np.median(results[feature]))

    print(condition, 'median slope: ', np.median(y_data_for_regression))
    median_slope = np.median(y_data_for_regression)
    xixs = [1,2,3,4,5,6]
    ax.hlines([median_slope,],
              xixs[0]-0.5, xixs[-1]+0.5,
              linestyles=['--',], colors=[MODEL_LINE_COLOR,])

    ax.set_xlim(0.5, 6.5)
    ax.set_ylim(0,8)

    ax.set_xticks(ticks)
    ax.set_yticks([0, 2, 4, 6, 8])
    if show_labels:
        ax.set_ylabel('Slope (rad/s)', labelpad=-2)
        ax.set_xlabel('Wind speed (cm/s)', labelpad=1)
        ax.set_xticklabels([0, 3, 10, 20, 30, 60])
        ax.set_yticklabels(['0', '', '', '', '8'])
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])

    ax.tick_params(axis='y', pad=2)

    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

    # data = diagnose_axis_elements(ax, verbose=True)
    set_selective_rasterization(ax, rasterize_markers=['o'])

flash median slope:  3.5956978903130628


sham median slope:  1.9051531472217134


In [17]:
feature = 'axis_ratios'

for condition in CONDITIONS:
    metadata = label_to_metadata.__getattribute__(condition)
    show_labels = (condition == CONDITIONS[0])  # labels only on the flash panel
    ax = layout.axes[('variable_wind_' + condition, 'axis_ratios')]

    x_data_for_regression = []
    y_data_for_regression = []

    ticks = []
    ticklabels = []
    for key, val in metadata.items():
        fname = key
        results = get_median_slope_and_axis_ratio(fname)
        xix, _, label, speed = val
        color = get_color(condition, label)

        ticks.append(xix)
        ticklabels.append(label)

        print(condition, key, len(results[feature]))
        hist_counts, hist_bins = np.histogram(results[feature], bins=20)
        fpl.scatter_box_violin_distribution(ax, xix, copy.copy(results[feature]), hist_counts, hist_bins, color=color, markersize=MARKERSIZE)

        if xix<=6:
            x_data_for_regression.append(xix)
            y_data_for_regression.append(np.median(results[feature]))

    result = scipy.stats.linregress(x_data_for_regression, y_data_for_regression)
    axis_ratio_slope = result.slope
    axis_ratio_intercept = result.intercept
    x = np.linspace(x_data_for_regression[0]-0.5, x_data_for_regression[-1]+0.5, 10)
    y = x*axis_ratio_slope + axis_ratio_intercept
    ax.plot(x, y, '--', color=MODEL_LINE_COLOR)

    print(condition, 'axis ratios', x_data_for_regression, y_data_for_regression)
    print(condition, 'axis_ratio_slope: ', axis_ratio_slope, ' axis_ratio_intercept: ', axis_ratio_intercept)

    ax.set_xlim(0.5, 6.5)
    ax.set_ylim(0,1)

    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    if show_labels:
        ax.set_ylabel('Axis ratio', labelpad=-2)
        ax.set_yticklabels(['0', '', '', '', '', '1'])
    else:
        ax.set_yticklabels([])

    ax.set_xticks(ticks)
    ax.set_xticklabels([])

    ax.tick_params(axis='y', pad=2)

    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)
    set_selective_rasterization(ax, rasterize_markers=['o'])

flash ../../../Data/unifying_algo_results/varwind_N700_flash_12.0_translationTrue.parquet 700
flash ../../../Data/unifying_algo_results/varwind_N700_flash_20.0_translationTrue.parquet 491


flash ../../../Data/unifying_algo_results/varwind_N700_flash_23.0_translationTrue.parquet 257
flash ../../../Data/unifying_algo_results/varwind_N700_flash_28.0_translationTrue.parquet 700


flash ../../../Data/unifying_algo_results/varwind_N700_flash_40.0_translationTrue.parquet 272
flash ../../../Data/unifying_algo_results/varwind_N700_flash_90.0_translationTrue.parquet 241
flash axis ratios [1, 2, 3, 4, 5, 6] [np.float64(0.682934699550005), np.float64(0.6365134696963373), np.float64(0.47095793849499484), np.float64(0.49548048641991316), np.float64(0.3575208154127715), np.float64(0.2389559723535617)]
flash axis_ratio_slope:  -0.086638544311657  axis_ratio_intercept:  0.7836288020787301
sham ../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-12-fan_0-cms_all-traj_align_SHAM.parquet 388


sham ../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-20-fan_0-03-cms_all-traj_align_SHAM.parquet 423
sham ../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-23-fan_0-1-cms_all-traj_align_SHAM.parquet 241
sham ../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-28-fan_0-2-cms_all-traj_align_SHAM.parquet 225
sham ../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-40-fan_0-3-cms_all-traj_align_SHAM.parquet 232
sham ../../../Data/unifying_algo_results/var_wind_kevin_analyzed/var-wind-90-fan_0-6-cms_all-traj_align_SHAM.parquet 68


sham axis ratios [1, 2, 3, 4, 5, 6] [np.float64(0.5732635934671366), np.float64(0.5683954024465927), np.float64(0.557841035419816), np.float64(0.5534130572210395), np.float64(0.5152869963013018), np.float64(0.4594786250538193)]
sham axis_ratio_slope:  -0.020933658248606737  axis_ratio_intercept:  0.6112142555217412


In [18]:
for condition in CONDITIONS:
    layout.append_figure_to_layer(layout.figures['variable_wind_' + condition], 'variable_wind_' + condition, cleartarget=True)
layout.write_svg(FIGURE_NAME)